# PredictIQ: Physics-Informed Predictive Maintenance with Explainable AI (XAI)
### Predicting CNC Milling Machine Failure using Machine Learning & SHAP

**Dataset:** AI4I 2020 Predictive Maintenance (UCI Machine Learning Repository)  
**Core Technologies:** Python, Pandas, Scikit-Learn, XGBoost, SHAP, Streamlit

---
## Project Overview & Objectives
Industrial unplanned downtime causes billions in lost revenue, ruined workpieces, and equipment damage.
Traditional maintenance follows two flawed paradigms:
1. **Reactive Maintenance:** Run-to-failure (expensive, hazardous).
2. **Preventive Maintenance:** Service on a fixed calendar (wasteful, throws away healthy components).

**PredictIQ** implements **Predictive Maintenance (PdM)**: Continuous sensor telemetry monitoring to predict impending failures *before* they manifest.



## Step 1: Environment Setup & Library Imports


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Add parent directory to sys.path for local src imports
sys.path.append('..')
from src.features import clean_raw_data, engineer_physics_features, prepare_single_sample
from src.explain import PredictIQExplainer

print('All libraries successfully imported!')



## Step 2: Data Ingestion & Exploratory Analysis
We load the **AI4I 2020** dataset (10,000 observations).
Each row records 14 attributes of a milling machine tool operation.



In [ ]:
data_path = '../data/ai4i2020.csv'
df_raw = pd.read_csv(data_path)

print(f'Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
df_raw.head()



In [ ]:
# Check data types and null values
print('Missing values per column:')
print(df_raw.isnull().sum())
print('\nClass Distribution for Machine failure:')
print(df_raw['Machine failure'].value_counts(normalize=True))



## Step 3: Resolving Target Leakage & Data Cleaning
### Critical Concept: Target Leakage
The columns `TWF`, `HDF`, `PWF`, `OSF`, and `RNF` are post-hoc failure diagnoses:
* `TWF` = Tool Wear Failure
* `HDF` = Heat Dissipation Failure
* `PWF` = Power Failure
* `OSF` = Overstrain Failure
* `RNF` = Random Failure

In a real operating factory, **sensors do NOT provide these labels beforehand**. Retaining them allows a model to cheat by reading the answer key. We must drop them along with record identifiers (`UDI`, `Product ID`).



In [ ]:
df_clean = clean_raw_data(df_raw)
print('Cleaned feature columns:')
print(df_clean.columns.tolist())



## Step 4: Physics-Informed Feature Engineering
Raw sensor telemetry alone misses complex thermal-mechanical interactions. We engineer 3 physical metrics:
1. **Temperature Difference (Delta T = T_process - T_air)**:
   Measures cooling efficiency. When Delta T < 8.6 K and speed is low, thermal dissipation fails.
2. **Mechanical Power (P = tau * omega)**:
   Power (Watts) = Torque [Nm] * (Rotational Speed [rpm] * 2*pi / 60)
   Identifies motor stall or electrical power surges.
3. **Overstrain Factor (S = Tool Wear * Torque)**:
   Measures the structural stress on the cutting insert.



In [ ]:
df_eng = engineer_physics_features(df_clean)
df_eng[['air_temp', 'process_temp', 'temp_diff', 'rot_speed', 'torque', 'power', 'tool_wear', 'wear_torque']].head()



## Step 5: Handling the 3.4% Class Imbalance
Failures represent only **3.39%** of the dataset.
* A dummy model predicting 0 gives **96.61% accuracy** but detects zero failures.
* Therefore, **Recall** is our chief priority: Recall = TP / (TP + FN).
* We use **Stratified 80/20 splitting** to maintain exact class proportions.



In [ ]:
from src.features import FEATURE_COLUMNS_RAW, FEATURE_COLUMNS_ENGINEERED

y = df_clean['failure']

# Baseline raw split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_clean[FEATURE_COLUMNS_RAW], y, test_size=0.20, random_state=42, stratify=y
)

# Engineered split
X_train_eng, X_test_eng, _, _ = train_test_split(
    df_eng[FEATURE_COLUMNS_ENGINEERED], y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training failures: {y_train.sum()} / {len(y_train)} ({y_train.mean():.2%})')
print(f'Testing failures : {y_test.sum()} / {len(y_test)} ({y_test.mean():.2%})')



## Step 6: Model Training & Experiment Benchmarking
We train 3 model families across both baseline and engineered feature sets:
1. **Logistic Regression** (Linear baseline with class_weight='balanced')
2. **Random Forest** (Bagging ensemble with class_weight='balanced')
3. **XGBoost** (Gradient boosting with scale_pos_weight)



In [ ]:
from src.train import evaluate_model

# Scalers for linear model
scaler_raw = StandardScaler().fit(X_train_raw)
scaler_eng = StandardScaler().fit(X_train_eng)

results = []

# 1. Logistic Regression
lr_b = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(scaler_raw.transform(X_train_raw), y_train)
results.append(evaluate_model('Logistic Regression (Baseline)', y_test, lr_b.predict(scaler_raw.transform(X_test_raw)), lr_b.predict_proba(scaler_raw.transform(X_test_raw))[:,1]))

lr_e = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(scaler_eng.transform(X_train_eng), y_train)
results.append(evaluate_model('Logistic Regression (Engineered)', y_test, lr_e.predict(scaler_eng.transform(X_test_eng)), lr_e.predict_proba(scaler_eng.transform(X_test_eng))[:,1]))

# 2. Random Forest
rf_b = RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42).fit(X_train_raw, y_train)
results.append(evaluate_model('Random Forest (Baseline)', y_test, rf_b.predict(X_test_raw), rf_b.predict_proba(X_test_raw)[:,1]))

rf_e = RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42).fit(X_train_eng, y_train)
results.append(evaluate_model('Random Forest (Engineered)', y_test, rf_e.predict(X_test_eng), rf_e.predict_proba(X_test_eng)[:,1]))

# 3. XGBoost
scale_pos = (len(y_train) - sum(y_train)) / sum(y_train)
xgb_b = XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.08, scale_pos_weight=scale_pos, eval_metric='logloss', random_state=42).fit(X_train_raw, y_train)
results.append(evaluate_model('XGBoost (Baseline)', y_test, xgb_b.predict(X_test_raw), xgb_b.predict_proba(X_test_raw)[:,1]))

xgb_e = XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.08, scale_pos_weight=scale_pos, eval_metric='logloss', random_state=42).fit(X_train_eng, y_train)
results.append(evaluate_model('XGBoost (Engineered)', y_test, xgb_e.predict(X_test_eng), xgb_e.predict_proba(X_test_eng)[:,1]))



## Step 7: Benchmark Results & Impact of Feature Engineering


In [ ]:
res_df = pd.DataFrame(results)[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
res_df



In [ ]:
# Visual comparison plot
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
metrics_to_plot = res_df.set_index('model')[['recall', 'f1']]

metrics_to_plot.plot(kind='barh', ax=ax[0], colormap='viridis')
ax[0].set_title('Recall vs F1-Score across Models', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Score')

# ROC Curves
from sklearn.metrics import roc_curve, auc
for m_name, model, x_data, is_scaled in [
    ('RF Engineered', rf_e, X_test_eng, False),
    ('XGB Engineered', xgb_e, X_test_eng, False),
    ('LR Engineered', lr_e, scaler_eng.transform(X_test_eng), True)
]:
    prob = model.predict_proba(x_data)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax[1].plot(fpr, tpr, label=f'{m_name} (AUC = {auc(fpr, tpr):.3f})')

ax[1].plot([0, 1], [0, 1], 'k--', alpha=0.6)
ax[1].set_title('ROC Curves (Engineered Features)', fontsize=12, fontweight='bold')
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate (Recall)')
ax[1].legend()
plt.tight_layout()
plt.show()



## Step 8: Model Explainability with SHAP (XAI)
To make PredictIQ actionable for plant engineers, we compute **SHAP (Shapley Additive exPlanations)**.
Let us examine an impending overstrain failure:



In [ ]:
explainer = PredictIQExplainer(xgb_e, X_train_eng.sample(100, random_state=42))

# Test with a hazardous condition: high torque (65 Nm) & high tool wear (220 min)
hazardous_sample = prepare_single_sample(
    air_temp=300.0,
    process_temp=310.0,
    rot_speed=1300.0,
    torque=65.0,
    tool_wear=220.0,
    machine_type='L'
)

prob = xgb_e.predict_proba(hazardous_sample)[0][1]
print(f'Predicted Failure Probability: {prob:.2%}')

exp, summary = explainer.explain_sample(hazardous_sample)
print(summary.head(6))

fig = explainer.plot_waterfall(exp)
plt.show()



## Step 9: Key Takeaways for Faculty & Viva
1. **Target Leakage Avoidance:** Dropping the post-mortem columns (`TWF`, `HDF`, etc.) avoided an artificial 100% test score and ensured realistic factory utility.
2. **Physics-Driven Gains:** Adding Delta T, P, and Overstrain boosted Random Forest Recall from **45.6% to 75.0%**, and XGBoost Recall reached **83.8%** with an F1 of **0.809**.
3. **Imbalance Strategy:** Stratified splitting paired with `scale_pos_weight` and `class_weight='balanced'` successfully penalized false negatives.
4. **Explainable AI:** SHAP provides transparency, pinpointing the physical mechanism (e.g., ToolWear x Torque) responsible for high risk scores.

